In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [3]:
# Path ke dataset
dataset_dir = "../dataset/Food-20"

img_height, img_width = 224, 224
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.3,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.3,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# Dari val_ds (30%), kamu bisa split lagi jadi val (20%) + test (10%)
val_batches = tf.data.experimental.cardinality(val_ds)
test_size = val_batches // 3  # kira-kira sepertiga jadi test

test_ds = val_ds.take(test_size)
val_ds = val_ds.skip(test_size)

# Class names
class_names = train_ds.class_names
print("Classes:", class_names)
print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Val batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Test batches:", tf.data.experimental.cardinality(test_ds).numpy())
# Base model: ResNet50 pretrained di ImageNet
base_model = tf.keras.applications.ResNet50(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model
base_model.trainable = False

# Build ResNet model
model_resnet = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(len(class_names), activation='softmax')
])

# Compile
model_resnet.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history_resnet = model_resnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

# Evaluate di test set
test_loss, test_acc = model_resnet.evaluate(test_ds, verbose=2)
print(f"\nTest Accuracy (ResNet50): {test_acc:.2f}, Test Loss: {test_loss:.2f}")

# Save model
model_resnet.save("../saved_models/resnet50.h5")
print("Model ResNet50 saved to models/resnet50.h5")


Found 2000 files belonging to 3 classes.
Using 1400 files for training.
Found 2000 files belonging to 3 classes.
Using 600 files for validation.
Classes: ['test', 'train', 'val']
Train batches: 44
Val batches: 13
Test batches: 6
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 56s 1us/step
Epoch 1/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - accuracy: 0.5885 - loss: 1.1038 - val_accuracy: 0.7010 - val_loss: 0.8810
Epoch 2/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 129s 3s/step - accuracy: 0.6966 - loss: 0.8218 - val_accuracy: 0.6446 - val_loss: 0.8691
Epoch 3/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 158s 3s/step - accuracy: 0.7145 - loss: 0.7525 - val_accuracy: 0.6985 - val_loss: 0.8348
Epoch 4/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 130s 3s/step - accuracy: 0.7333 - loss: 0.6686 - val_accuracy: 0.6373 - val_loss: 0.8955
Epoch 5/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 129s 3s/step - accuracy: 0.7458 - loss: 0.6420 - val_accuracy: 0.6789 - val_loss: 0.9310
Epoch 6/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 102s 2s/step - accuracy: 0.7445 - loss: 0.6184


Test Accuracy (ResNet50): 0.68, Test Loss: 0.92
Model ResNet50 saved to models/resnet50.h5
